# CNN Example 1
For this example, we have images of cars and flowers, which have been divided into training and testing sets, and we have to build a CNN that identifies whether an image is a car or a flower.

Use car_flower_small folder

### Step 1: Import the numpy library and the necessary Keras libraries and classes

In [1]:
# Import the Libraries
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPool2D
from keras.layers import Flatten
from keras.layers import Dense
import numpy as np
from tensorflow import random

### Step 2: Now, set a seed and initiate the model with the `Sequential` class

In [2]:
#set a seed
seed = 1
np.random.seed(seed)
random.set_seed(seed)

# Initialising the CNN
classifier = Sequential()

### Step 3: Add the first layer of the CNN, set the input shape to (64, 64, 3), the dimension of each image, and set the activation function as a ReLU:

In [4]:
classifier.add(
    Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation='relu',
        input_shape=(64, 64, 3)
    )
)

/opt/anaconda3/envs/IOD2/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Step 4: Now, add the pooling layer with the image size as 2x2

In [5]:
classifier.add(MaxPool2D(pool_size=(2, 2)))

### Step 5: Flatten the output of the pooling layer by adding a flattening layer to the CNN model:

In [6]:
classifier.add(Flatten())

### Step 6: Add the first Dense layer of the MLP. 
Here, 128 is the output of the number of nodes. As a good practice, 128 is good to get started. activation is relu. As a good practice, the power of two is preferred

In [7]:
classifier.add(Dense(units=128, activation='relu'))

### Step 7: Add the output layer of the MLP.
This is a binary classification problem, so the size is 1 and the activation is `sigmoid`:

In [8]:
classifier.add(Dense(units=1, activation='sigmoid'))

### Step 8: Compile the network
Use an adam optimizer and compute the accuracy during the training process 

In [9]:
classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Step 9: Create training and test data generators. 
- Rescale the training and test images by `1/255` so that all the values are between `0` and `1`.
- Set these parameters for the training data generators only 
 - `shear_range=0.2`, `zoom_range=0.2`, and `horizontal_flip=True`
 
 - https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

ADDED STEP - Creating labels dataframe

In [18]:
from PIL import Image
import os

image_dir = "car_flower_small"

valid_files = []

for f in os.listdir(image_dir):
    if not f.lower().endswith(".jpg"):
        continue

    file_path = os.path.join(image_dir, f)

    try:
        with Image.open(file_path) as img:
            img.verify()  # checks if image is readable
        valid_files.append(f)
    except Exception:
        print(f"❌ Skipping unreadable image: {f}")

import pandas as pd

df = pd.DataFrame({"filename": valid_files})

df["class"] = df["filename"].apply(
    lambda x: "car" if "car" in x.lower() else "flower"
)

df["class"].value_counts()


❌ Skipping unreadable image: car.1115.jpg


class
flower    999
car       998
Name: count, dtype: int64

### Step 10: Create a training set from the training set folder.
'training_set' is the folder where our data has been placed. Our CNN model has an image size of `64x64`, so the same size should be passed here too. `batch_size` is the number of images in a single batch, which is `32`. `Class_mode` is set to binary since we are working on binary classifiers

In [19]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["class"]
)

### Step 11: Repeat step 10 for the test set 
while setting the folder to the location of the test images, that is, 'test_set'

In [20]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

### Step 12: Finally, fit the data. 
Set the `steps_per_epoch` to `STEP_SIZE_TRAIN` and the `validation_steps` to `STEP_SIZE_TEST`. 

Why do we need `steps_per_epoch` ?

Keep in mind that a Keras data generator is meant to loop infinitely — it should never return or exit.

Since the function is intended to loop infinitely, Keras has no ability to determine when one epoch starts and a new epoch begins.

Therefore, we compute the `steps_per_epoch` value as the total number of training data points divided by the batch size. Once Keras hits this step count it knows that it’s a new epoch.

Found 1997 validated image filenames.


In [21]:
training_set = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=image_dir,
    x_col="filename",
    y_col="class",
    target_size=(64, 64),
    batch_size=32,
    class_mode="binary"
)

test_set = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=image_dir,
    x_col="filename",
    y_col="class",
    target_size=(64, 64),
    batch_size=32,
    class_mode="binary"
)

Found 1597 validated image filenames belonging to 2 classes.
Found 400 validated image filenames belonging to 2 classes.


In [22]:
classifier.fit(
    training_set,
    epochs=10,
    validation_data=test_set
)

Epoch 1/10
 3/50 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.6929 - loss: 0.6130

/opt/anaconda3/envs/IOD2/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.7089 - loss: 0.5816 - val_accuracy: 0.7725 - val_loss: 0.5056
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7492 - loss: 0.4935 - val_accuracy: 0.7600 - val_loss: 0.5038
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8046 - loss: 0.4203 - val_accuracy: 0.7650 - val_loss: 0.5246
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8154 - loss: 0.4120 - val_accuracy: 0.7975 - val_loss: 0.4705
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8247 - loss: 0.4013 - val_accuracy: 0.7950 - val_loss: 0.4846
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8235 - loss: 0.3891 - val_accuracy: 0.7700 - val_loss: 0.5337
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8458 - loss: 0.3435 - val_accuracy: 0.7950 - val_loss: 0.4943
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8385 - loss: 0.3833 - val_accuracy: 0.7925 - val_loss: 0.

ADDED STEP - Adding on to dataframe with results

In [24]:
# ADDED
predict_datagen = ImageDataGenerator(rescale=1./255)

predict_generator = predict_datagen.flow_from_dataframe(
    dataframe=df,
    directory=image_dir,
    x_col="filename",
    y_col=None,          # no labels needed for prediction
    target_size=(64, 64),
    batch_size=32,
    shuffle=False,
    class_mode=None
)

Found 1997 validated image filenames.


In [26]:
import numpy as np

pred_probs = classifier.predict(predict_generator)
pred_probs = pred_probs.flatten()


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


In [27]:
df["pred_prob_flower"] = pred_probs

df["predicted_class"] = np.where(
    df["pred_prob_flower"] >= 0.5,
    "flower",
    "car"
)

In [28]:
df["correct"] = df["predicted_class"] == df["class"]


In [29]:
df.head()
df["correct"].value_counts(normalize=True)


correct
True     0.885829
False    0.114171
Name: proportion, dtype: float64

In [35]:
df

,filename,class,pred_prob_flower,predicted_class,correct
0,flower.1965.jpg,flower,0.985475,flower,True
1,car.718.jpg,car,0.461711,car,True
2,flower.441.jpg,flower,0.672173,flower,True
3,car.2543.jpg,car,0.013925,car,True
4,car.3885.jpg,car,0.005217,car,True
...,...,...,...,...,...
1992,flower.1592.jpg,flower,0.985530,flower,True
1993,flower.646.jpg,flower,0.913738,flower,True
1994,flower.120.jpg,flower,0.653207,flower,True
1995,flower.2919.jpg,flower,0.922512,flower,True


In [42]:
correct_df = df[df['correct'] == True]
correct_df.nlargest(10, 'pred_prob_flower')

,filename,class,pred_prob_flower,predicted_class,correct
1419,flower.4600.jpg,flower,1.000000,flower,True
324,flower.4996.jpg,flower,0.999999,flower,True
1942,flower.3185.jpg,flower,0.999999,flower,True
507,flower.736.jpg,flower,0.999997,flower,True
1216,flower.696.jpg,flower,0.999995,flower,True
1612,flower.3782.jpg,flower,0.999995,flower,True
82,flower.3627.jpg,flower,0.999984,flower,True
1256,flower.655.jpg,flower,0.999975,flower,True
1312,flower.4837.jpg,flower,0.999974,flower,True
1409,flower.2503.jpg,flower,0.999968,flower,True


In [38]:
incorrect_df = df[df['correct'] == False]
incorrect_df.nlargest(10, 'pred_prob_flower')

,filename,class,pred_prob_flower,predicted_class,correct
54,car.3846.jpg,car,0.999999,flower,False
734,car.1934.jpg,car,0.997351,flower,False
1355,car.3673.jpg,car,0.981203,flower,False
1245,car.4332.jpg,car,0.972296,flower,False
1880,car.284.jpg,car,0.971666,flower,False
847,car.5159.jpg,car,0.970970,flower,False
996,car.684.jpg,car,0.966083,flower,False
193,car.5166.jpg,car,0.960614,flower,False
565,car.5013.jpg,car,0.957384,flower,False
1248,car.2025.jpg,car,0.953395,flower,False


In [39]:
incorrect_df['pred_prob_flower'].median()

np.float32(0.57309884)

In [43]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df["class"],
    df["predicted_class"],
    labels=["car", "flower"]
)

cm

array([[860, 138],
       [ 90, 909]])